In [4]:
import torch

import torch.nn as nn
from torch.nn import functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

block_size = 8
batch_size = 4

learning_rate = 3e-4

eval_iters = 100


max_iters = 1000


cpu


In [5]:
@torch.no_grad()
def estimate_loss():
    out = {}

    model.eval()

    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)

        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()

        out[split] = losses.mean()

    model.train()

    return out

In [32]:
losses = estimate_loss()

print("Training loss:", losses['train'].item())
print("Validation loss:", losses['val'].item())

Training loss: 4.392825126647949
Validation loss: 4.371686935424805


In [7]:
with open('The_Odyssey.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(set(text))
print(chars)
vocab_size = len(chars)

['\n', ' ', '!', '(', ')', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'R', 'S', 'T', 'U', 'V', 'W', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '—', '’', '“', '”']


In [8]:
string_to_int = {ch:i for i,ch in enumerate(chars)}
int_to_string = {i:ch for i,ch in enumerate(chars)}

encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)

print(data[:100])



tensor([39, 52, 49,  1, 35, 48, 69, 63, 63, 49, 69,  0,  0, 46, 69,  1, 28, 59,
        57, 49, 62,  0,  0,  0, 23, 59, 58, 64, 49, 58, 64, 63,  0,  0,  1, 39,
        28, 25,  1, 35, 24, 43, 38, 38, 25, 43,  0,  1, 22, 35, 35, 31,  1, 29,
         7,  0,  0, 39, 28, 25,  1, 35, 24, 43, 38, 38, 25, 43,  0,  0,  0, 22,
        35, 35, 31,  1, 29,  0,  0,  0, 39, 28, 25,  1, 27, 35, 24, 38,  1, 29,
        34,  1, 23, 35, 40, 34, 23, 29, 32, 71])


In [9]:
n = int(0.8 * len(data))
train_data = data[:n]
val_data = data[n:]

In [10]:
block_size = 8

x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print('when input is', context, 'target is', target)

when input is tensor([39]) target is tensor(52)
when input is tensor([39, 52]) target is tensor(49)
when input is tensor([39, 52, 49]) target is tensor(1)
when input is tensor([39, 52, 49,  1]) target is tensor(35)
when input is tensor([39, 52, 49,  1, 35]) target is tensor(48)
when input is tensor([39, 52, 49,  1, 35, 48]) target is tensor(69)
when input is tensor([39, 52, 49,  1, 35, 48, 69]) target is tensor(63)
when input is tensor([39, 52, 49,  1, 35, 48, 69, 63]) target is tensor(63)


In [11]:
def get_batch(split):
    data = train_data if split == 'train' else val_data

    ix = torch.randint(len(data) - block_size, (batch_size,))

    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])

    x, y = x.to(device), y.to(device)

    return x, y

In [12]:
x, y = get_batch('train')

print('inputs:')
print(x)

print('targets:')
print(y)

inputs:
tensor([[58,  1, 64, 59,  1, 46, 49, 45],
        [45, 67, 45, 69,  1, 57, 59, 62],
        [63,  1, 56, 53, 66, 49, 63,  1],
        [67,  1, 57, 45, 58, 69,  1, 45]])
targets:
tensor([[ 1, 64, 59,  1, 46, 49, 45, 62],
        [67, 45, 69,  1, 57, 59, 62, 49],
        [ 1, 56, 53, 66, 49, 63,  1, 64],
        [ 1, 57, 45, 58, 69,  1, 45, 58]])


In [13]:
xb, yb = get_batch('train')

print("Input shape:", xb.shape)
print("Target shape:", yb.shape)

print("\nFirst input sequence:")
print(xb[0])

print("\nFirst target sequence:")
print(yb[0])

print("\nDecoded input:")
print(decode(xb[0].tolist()))

print("\nDecoded target:")
print(decode(yb[0].tolist()))

Input shape: torch.Size([4, 8])
Target shape: torch.Size([4, 8])

First input sequence:
tensor([50,  1, 39, 62, 59, 69,  7,  1])

First target sequence:
tensor([ 1, 39, 62, 59, 69,  7,  1, 33])

Decoded input:
f Troy. 

Decoded target:
 Troy. M


In [14]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [15]:
vocab_size = len(chars)

print(vocab_size)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, index, targets=None):
        logits = self.token_embedding_table(index)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, index, max_new_tokens):
        for _ in range(max_new_tokens):

            # Get predictions
            logits, loss = self.forward(index)

            # Focus only on the final time step
            logits = logits[:, -1, :]

            # Convert logits into probabilities
            probs = F.softmax(logits, dim=-1)

            # Sample the next token
            index_next = torch.multinomial(
                probs,
                num_samples=1
            )

            # Append the sampled token to the sequence
            index = torch.cat(
                (index, index_next),
                dim=1
            )

        return index

75


In [16]:
model = BigramLanguageModel(vocab_size)
m = model.to(device)

context = torch.zeros(
    (1, 1),
    dtype=torch.long,
    device=device
)

generated_chars = decode(
    m.generate(
        context,
        max_new_tokens=500
    )[0].tolist()
)

print(generated_chars)


(—NNT
’tFq
“8!”
’9VC2Hn0BAjWc”WwP.yZ-,vVvtjrN30LOcbwvI)r?,PhYJH3?—EeG“’ CM-!.;ADn6k,RsdOdPsAq“”  6DCUVJC0rPN5pVFi7mM 9H“)rIjwq8?YWe9Ye.3-:ulJLE““p::.6l—--kmB2—-Umo,IbNoYf’5rvostkJN4ZKVu“z1sK8“J’eC;Bg?u5g68?—6O(,I.0 9AElEsd;7;yZGs—7sA.3MxDjP,,Ir5Ck”j:p1”A(Vzl2’SLfEW!iL
’e0iSnMf.eCxc’e4u2EDM39hd;iG—8PDnShUG9T-4kdHVazG3bNa7!Wlk
M7j“Wmc7mPhtMgu22’CKxTm(sG6P0af5ZJ7“:NmOIPGwGslwTPhzTx—Et1Odt-8Vt)Ph-mwC2’m(U(0WcwmL
’FT;t7yb;b8)fV“4Yi95:48?hk0,jNO8? db?”5-4c.WN,lUVFT.7“p?m1Ae’SVslBIv—8PeP U”Poq—lPhPwJ2W


In [17]:
model = BigramLanguageModel(vocab_size)

xb, yb = get_batch('train')

logits, loss = model(xb, yb)

print(logits.shape)
print(loss)

torch.Size([32, 75])
tensor(4.8639, grad_fn=<NllLossBackward0>)


In [18]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate
)

for iter in range(max_iters):

    if iter % eval_iters == 0:
        losses = estimate_loss()

        print(
            f"step: {iter}, "
            f"train loss: {losses['train']:.3f}, "
            f"val loss: {losses['val']:.3f}"
        )

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model.forward(xb, yb)

    optimizer.zero_grad(set_to_none=True)

    loss.backward()

    optimizer.step()

print(loss.item())

step: 0, train loss: 4.883, val loss: 4.879
step: 100, train loss: 4.865, val loss: 4.910
step: 200, train loss: 4.840, val loss: 4.813
step: 300, train loss: 4.829, val loss: 4.836
step: 400, train loss: 4.742, val loss: 4.759
step: 500, train loss: 4.750, val loss: 4.795
step: 600, train loss: 4.718, val loss: 4.753
step: 700, train loss: 4.703, val loss: 4.716
step: 800, train loss: 4.701, val loss: 4.677
step: 900, train loss: 4.621, val loss: 4.665
4.923325538635254


In [31]:
context = torch.zeros(
    (1, 1),
    dtype=torch.long,
    device=device
)

generated_chars = decode(
    m.generate(
        context,
        max_new_tokens=500
    )[0].tolist()
)

print(generated_chars)


dF:p-”Izt
V9URI”
.5b-4V2pL8l4sa5)“HnPglEjqvVzfbH“HKYRp6knlWx”cf,rhR6 9y5b8UWFI“wtWCUJM?qawv3vE8?I1VKrrPwV9H2NmV5bh;
..Z—v)8?j2wrA9axD6?KN,6’Kk8:.,80 d,uWhUPBGrrp-R5-“g7vmweww LTYKu MHI6k1iw8!6tT’n0zY.5fV65xNOsv”vd
’FG“”.3KWxcmHbN7N—--3MC)H8
J,GSD0E’’g(UGVGs689!Kqg)“-.8?IBlV21y:pRUSybvMLPhWA3f.rtHax6aOsI’V6vRSwE:Yevm)7bBGGFV2DKeE46-4UGbguMIN-gpsOLp8VGxaR)FDt-8?Jzr:Se
qP
 uV2NRv9eh7Kq- BDM”k?ZxqG;w-x”sr’diVC6J;Z CchRvTKA.H2V9JfUV?vkk?—“Y.R5JZb8tt0K2N:Ewv)“zFW““.ZPp:kmR:d—0K!zNV;7MaMbeqR9-11iKkkkst


In [19]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate
)

for iter in range(1000):

    # Sample a batch of training data
    xb, yb = get_batch('train')

    # Forward pass
    logits, loss = model(xb, yb)

    # Clear gradients from the previous iteration
    optimizer.zero_grad(set_to_none=True)

    # Calculate gradients using backpropagation
    loss.backward()

    # Update the model parameters
    optimizer.step()

print(loss.item())

4.180601596832275


In [20]:
for iter in range(100):xb, yb = get_batch('train')

logits, loss = model(xb, yb)

optimizer.zero_grad(set_to_none=True)

loss.backward()

optimizer.step()

print(loss.item())

4.624256134033203


In [21]:
xb, yb = get_batch('train')

logits = model.token_embedding_table(xb)

print("Original logits shape:", logits.shape)
print("Original targets shape:", yb.shape)

B, T, C = logits.shape

reshaped_logits = logits.view(B*T, C)
reshaped_targets = yb.view(B*T)

print("Reshaped logits:", reshaped_logits.shape)
print("Reshaped targets:", reshaped_targets.shape)

Original logits shape: torch.Size([4, 8, 75])
Original targets shape: torch.Size([4, 8])
Reshaped logits: torch.Size([32, 75])
Reshaped targets: torch.Size([32])


In [22]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)

generated = model.generate(
    context,
    max_new_tokens=100
)

print(decode(generated[0].tolist()))


—gO(c “lO’B(GSe:HkJUYxEAagjm?(1cW:x7yxfacVpj”2KCRFsfk
z3M?ii-xFVK4kj14MaEkgO,.e
 T-B,C—VIEf,kysFD?3z


In [23]:
print(hasattr(model, "generate"))

True


In [24]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)

generated = model.generate(
    context,
    max_new_tokens=100
)

print(decode(generated[0].tolist()))


tk:Dq;xbHFDhag7kIuF2r,Nbx0:LB6—LzFnSyKAbHVpEkyoawhNCI7;mUAb3WCFcVEbEgKj”-—z ke,9sx95O!2A?ReCmvZiP;xf


In [25]:
print(hasattr(model, "generate"))

True


In [26]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)

generated = model.generate(
    context,
    max_new_tokens=100
)

print(decode(generated[0].tolist()))


R0B8B2r0“Ikxf8yYo wh:q37HObyAuGl;“8j.guiJWm 9-m)wBPUgho FndS ,”Z 9M:2Hn“qz,mzs0F09P50;xjPbo’E3P,Fn“0


In [27]:
test_context = torch.zeros((1, 8), dtype=torch.long, device=device)

test_logits, _ = model(test_context)

print("Before selecting last position:", test_logits.shape)

last_logits = test_logits[:, -1, :]

print("After selecting last position:", last_logits.shape)

Before selecting last position: torch.Size([1, 8, 75])
After selecting last position: torch.Size([1, 75])


In [28]:
for iter in range(100):
    xb, yb = get_batch('train')

    # Forward pass
    logits, loss = model(xb, yb)

    # Clear gradients from previous iteration
    optimizer.zero_grad(set_to_none=True)

    # Backpropagation
    loss.backward()

    # Update model parameters
    optimizer.step()

print(loss.item())

4.062831878662109


In [29]:
@torch.no_grad()
def estimate_loss():
    out = {}

    model.eval()

    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)

        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()

        out[split] = losses.mean()

    model.train()

    return out

In [30]:
losses = estimate_loss()

print("Training loss:", losses['train'].item())
print("Validation loss:", losses['val'].item())

Training loss: 4.376849174499512
Validation loss: 4.376020908355713
